# VANTRA OCR — GPU hyperparameter sweep (Indian domain)

**What this does:** fine-tunes the EasyOCR CRNN recognizer on the leak-free Indian
plate corpus (train=1,295 imgs/765 plates + 3x aug = 5,180 rows; val=100; test=312,
partitioned by unique plate text) with a proper learning-rate × epoch sweep on GPU.
Optionally continues from the r3 (international production) checkpoint on a mixed
Indian+international corpus. Optional final experiment: PARSeq-style transformer
recognizer.

## Instructions
1. **Runtime → Change runtime type → T4 GPU** before running anything.
2. Upload `colab_package.zip` (140MB: Indian splits + r3 checkpoint) to the Colab
   file browser (drag into the sidebar). The international train **images** are
   NOT in the zip (1.9GB) — only their labels. The notebook detects this:
   - Without int'l images: runs the Indian-only sweep (still the core experiment).
   - With int'l images (upload `ft_train_images.zip` containing 90,744 PNGs, or
     mount Drive with the VANTRA repo and set INTL_IMG_DIR): also runs the
     full combined-corpus sweep from r3.
3. Run all cells (Runtime → Run all). The sweep takes ~15-30 min on a T4.
4. When done, download `best_checkpoints.zip` from the file browser — it contains
   the best checkpoint(s) by validation exact match, plus `sweep_summary.csv`.


In [ ]:
# 1. Verify GPU — STOP if none
import subprocess, sys
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout[-800:] if r.stdout else r.stderr)
if r.returncode != 0:
    print('\n*** NO GPU DETECTED — Runtime > Change runtime type > T4 GPU, then restart. ***')
    sys.exit(1)

In [ ]:
# 2. Unpack data
!unzip -q -o colab_package.zip -d /content/data
!ls -la /content/data /content/data/ft_indian

In [ ]:
# 3. Install deps
!pip -q install easyocr torch pillow

In [ ]:
# 4. Setup: load model, define dataset/training (GPU-native port of the local trainer)
import math, random, json, time, sys, os
from pathlib import Path
import torch, torch.nn as nn
import numpy as np
from PIL import Image

DATA = Path('/content/data/ft_indian')
R3 = Path('/content/data/r3_recognizer.pt')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'GPU required — see cell 1'

import easyocr
from easyocr.utils import CTCLabelConverter
from easyocr.recognition import NormalizePAD

reader = easyocr.Reader(['en'], gpu=True, verbose=False, quantize=False)
BASE_MODEL = reader.recognizer
CONVERTER = CTCLabelConverter(reader.character)

# --- optional international images ---
INTL_LABELS = Path('/content/data/intl_train_labels.txt')
INTL_IMG_DIR = Path('/content/ft_train_images')  # set if images uploaded elsewhere
intl_available = INTL_LABELS.exists() and (INTL_IMG_DIR.exists() or Path('/content/data/ft/train').exists())
intl_img_root = INTL_IMG_DIR if INTL_IMG_DIR.exists() else Path('/content/data/ft/train')
print('international images available:', intl_available, '->', intl_img_root if intl_available else '')

def load_indian(split):
    out = []
    for line in (DATA / f'in_{split}' / 'labels.txt').read_text().splitlines():
        name, label = line.split('\t')
        out.append((DATA / f'in_{split}' / name, label))
    return out

def load_combined():
    out = []
    for line in (DATA / 'in_train' / 'labels.txt').read_text().splitlines():
        name, label = line.split('\t')
        out.append((DATA / 'in_train' / name, label))
    if intl_available:
        for line in INTL_LABELS.read_text().splitlines():
            name, label = line.split('\t')
            out.append((intl_img_root / name, label))
    return out

class PlateDataset(torch.utils.data.Dataset):
    def __init__(self, items, imgH=64, max_w=480):
        self.items, self.imgH, self.max_w = items, imgH, max_w
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        img = Image.open(path).convert('L')
        w, h = img.size
        rw = min(self.max_w, max(16, math.ceil(self.imgH * (w / float(h)))))
        resized = img.resize((rw, self.imgH), Image.BICUBIC)
        return NormalizePAD((1, self.imgH, self.max_w))(resized), label

def collate(b):
    return torch.stack([x[0] for x in b]), [x[1] for x in b]

def encode(converter, labels, device):
    idxs, lengths = [], []
    for lab in labels:
        seq = [converter.dict[c] for c in lab if c in converter.dict]
        enc, prev = [], None
        for c in seq:
            if c == prev:
                enc.append(0)
            enc.append(c)
            prev = c
        idxs.extend(enc)
        lengths.append(len(enc))
    return torch.IntTensor(idxs).to(device), torch.IntTensor(lengths).to(device)

def lev(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca != cb)))
        prev = cur
    return prev[-1]

@torch.no_grad()
def evaluate(model, items, bs=64):
    model.eval()
    exact = cc = nc = 0
    for i in range(0, len(items), bs):
        chunk = items[i:i+bs]
        ds = PlateDataset(chunk)
        tensors, labels = collate([ds[j] for j in range(len(chunk))])
        tensors = tensors.to(DEVICE)
        bml = tensors.size(3) // 10
        tfp = torch.LongTensor(tensors.size(0), bml + 1).fill_(0).to(DEVICE)
        preds = model(tensors, tfp).log_softmax(2)
        _, idx = preds.max(2)
        idx = idx.cpu().numpy()
        for b in range(tensors.size(0)):
            seq, prev = [], 0
            for t in idx[b]:
                if t != prev and t != 0:
                    seq.append(t)
                prev = t
            text = ''.join(CONVERTER.character[c] for c in seq)
            tgt = labels[b]
            if text == tgt:
                exact += 1
            cc += max(0, len(tgt) - lev(text, tgt))
            nc += len(tgt)
    n = len(items)
    return {'exact_pct': 100*exact/n, 'char_pct': 100*cc/max(nc,1)}

def train(tag, lr, epochs, train_items, init_ckpt=None, bs=32):
    model = easyocr.Reader(['en'], gpu=True, verbose=False, quantize=False).recognizer
    if init_ckpt:
        model.load_state_dict(torch.load(init_ckpt, map_location='cpu', weights_only=False))
    model.to(DEVICE).train()
    ti = [(p,l) for p,l in train_items if all(c in CONVERTER.dict for c in l)]
    random.Random(0).shuffle(ti)
    vi = [(p,l) for p,l in VAL if all(c in CONVERTER.dict for c in l)]
    loader = torch.utils.data.DataLoader(PlateDataset(ti), batch_size=bs, shuffle=True, collate_fn=collate)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    ctc = nn.CTCLoss(blank=0, zero_infinity=True)
    best, hist = -1, []
    for ep in range(epochs):
        model.train()
        for tensors, labels in loader:
            tensors = tensors.to(DEVICE)
            bml = tensors.size(3) // 10
            tfp = torch.LongTensor(tensors.size(0), bml+1).fill_(0).to(DEVICE)
            preds = model(tensors, tfp).log_softmax(2).permute(1,0,2)
            B, T, C = preds.size()
            tgt, tl = encode(CONVERTER, labels, DEVICE)
            il = torch.full((B,), T, dtype=torch.long, device=DEVICE)
            loss = ctc(preds, tgt, il, tl)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        v = evaluate(model, vi)
        v['epoch'] = ep+1
        hist.append(v)
        print(f'  [{tag}] ep{ep+1}: val exact {v["exact_pct"]:.1f}% char {v["char_pct"]:.1f}%', flush=True)
        if v['exact_pct'] >= best:
            best = v['exact_pct']
            torch.save(model.state_dict(), f'/content/ckpt_{tag}.pt')
    return best, hist

VAL = load_indian('val')
print('val items:', len(VAL))
INDIAN_TRAIN = load_indian('train')
print('indian train rows (incl. aug):', len(INDIAN_TRAIN))

In [ ]:
# 5. SWEEP A — Indian-only: lr x epochs grid (the core experiment)
sweep_a = []
for lr in [1e-4, 5e-5, 2e-5]:
    for epochs in [12, 20]:
        tag = f'indA_lr{lr}_e{epochs}'
        print(f'\n=== {tag} ===')
        best, hist = train(tag, lr, epochs, INDIAN_TRAIN)
        sweep_a.append({'config': tag, 'lr': lr, 'epochs': epochs, 'best_val_exact': best, 'history': hist})
print('\nSweep A done.')

In [ ]:
# 6. SWEEP B — from r3 on FULL combined corpus (only if int'l images present)
sweep_b = []
if intl_available:
    COMBINED = load_combined()
    print('combined rows:', len(COMBINED))
    for lr in [1e-4, 5e-5]:
        tag = f'indB_lr{lr}'
        print(f'\n=== {tag} (3 epochs, from r3) ===')
        best, hist = train(tag, lr, 3, COMBINED, init_ckpt=str(R3))
        sweep_b.append({'config': tag, 'lr': lr, 'epochs': 3, 'best_val_exact': best, 'history': hist})
else:
    print('International images not present — skipping combined sweep (Indian-only sweep above is the core result).')

## Optional: PARSeq-style transformer recognizer (separate experiment)

**This cell is independent of the sweep above.** It installs the `parseq` package
and trains a small transformer OCR model from scratch on the Indian corpus.
If it fails or runs long, skip it — the CRNN sweep above is the deliverable.


In [ ]:
# 7. OPTIONAL — PARSeq transformer recognizer (experimental, safe to skip)
RUN_PARSEQ = False  # set True to attempt
if RUN_PARSEQ:
    !pip -q install parseq
    # PARSeq expects a different data format (LMDB); converting + training is
    # non-trivial — follow https://github.com/baudm/parseq for the training
    # recipe. This stub exists to mark the path; do not block on it.
    print('PARSeq path documented but not wired — see parseq repo.')
else:
    print('Skipped PARSeq (RUN_PARSEQ=False).')

In [ ]:
# 8. Summary + final test evaluation + package best checkpoints
import csv, shutil

all_runs = sweep_a + sweep_b
print('='*70)
print(f'{"config":28s} {"best val exact":>14s}')
print('-'*70)
for r in all_runs:
    print(f'{r["config"]:28s} {r["best_val_exact"]:>13.1f}%')
best_run = max(all_runs, key=lambda r: r['best_val_exact'])
print('='*70)
print('BEST:', best_run['config'])

# one-shot test eval of the best checkpoint
TEST = load_indian('test')
TEST = [(p,l) for p,l in TEST if all(c in CONVERTER.dict for c in l)]
m = easyocr.Reader(['en'], gpu=True, verbose=False, quantize=False).recognizer
m.load_state_dict(torch.load(f'/content/ckpt_{best_run["config"]}.pt', map_location='cpu', weights_only=False))
m.to(DEVICE)
res = evaluate(m, TEST)
print(f'\nBEST MODEL on leak-free Indian test (n={len(TEST)}): exact {res["exact_pct"]:.1f}% char {res["char_pct"]:.1f}%')
print('(reference: inB1_sub deployed locally = 45.8% exact / 84.8% char; inA2 = 53.2%/87.5%)')

with open('/content/sweep_summary.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['config', 'lr', 'epochs', 'best_val_exact', 'test_exact', 'test_char'])
    for r in all_runs:
        w.writerow([r['config'], r['lr'], r['epochs'], r['best_val_exact'],
                    res['exact_pct'] if r is best_run else '', res['char_pct'] if r is best_run else ''])
shutil.make_archive('/content/best_checkpoints', 'zip', '/content',
                    *(f for f in os.listdir('/content') if f.startswith('ckpt_') or f == 'sweep_summary.csv'))
# simpler: repack explicitly
os.system('cd /content && zip -q -j best_checkpoints.zip ckpt_*.pt sweep_summary.csv')
print('\n>>> Download best_checkpoints.zip from the file browser (folder icon, left sidebar). <<<')